## GOLD LAYER - AGGREGATIONS  TABLES

### Purpose
- Build business-ready aggregated tables for reporting and dashboards  
- Provide meaningful KPIs derived from Gold fact and dimension tables  
- Reduce complexity for analytics consumers by precomputing common metrics  

### Aggregations Included
1. **Monthly Sales Trend**
   - Tracks month-wise sales, transactions, discounts, and average order value  

2. **Top 3 Menu Items by Sales**
   - Identifies the best performing products by total revenue and quantity sold  

3. **Voucher Usage Summary**
   - Compares voucher vs non-voucher transactions and their sales/discount impact  

4. **Payment Method Split**
   - Shows customer payment preference distribution and sales contribution per method  

5. **Customer Lifetime Value (CLV)**
   - Calculates customer spend, order count, average order value, and first/last purchase date  

6. **Store Performance**
   - Measures store-wise sales, transaction count, average order value, and discounts  
   - Includes region-level attributes such as city and state  

7. **Top 10 Customers by Spend**
   - Identifies the highest value customers based on total spend  

### Notes
- Dimension tables are maintained as **SCD Type 2**  
- Current dimension records are selected using:
  - `__END_AT IS NULL`


In [0]:
------------------------------------
-- AGG 1: MONTHLY SALES TREND
--
-- Business KPI:
--   - Total sales per month
--   - Total transactions per month
--   - Total discount per month
--   - Average order value per month
-------------------------------------
CREATE OR REFRESH MATERIALIZED VIEW coffee.gold.agg_monthly_sales
COMMENT "Monthly sales KPIs derived from fact_transactions"
AS
SELECT
  date_trunc('month', created_at) AS sales_month,
  COUNT(DISTINCT transaction_id) AS total_transactions,
  ROUND(SUM(final_amount), 2) AS total_sales,
  ROUND(SUM(discount_applied), 2) AS total_discount,
  ROUND(AVG(final_amount), 2) AS avg_order_value
FROM coffee.gold.fact_transactions
GROUP BY date_trunc('month', created_at);


In [0]:
--------------------------------------------------------
-- AGG 2: TOP 3 MENU ITEMS BY SALES
--
-- Business KPI:
--   - Identify top performing products
--   - Total quantity and revenue per item
--
-- Notes:
--   - Join to dim_menu_items to get item_name and category
--------------------------------------------------------
CREATE OR REFRESH  MATERIALIZED VIEW  coffee.gold.agg_top_3_menu_items
COMMENT "Top 3 menu items by sales (subtotal) derived from fact_transaction_items"
AS
SELECT
  ti.item_id,
  mi.item_name,
  mi.category,
  SUM(ti.quantity) AS total_quantity_sold,
  ROUND(SUM(ti.subtotal), 2) AS total_sales
FROM coffee.gold.fact_transaction_items ti
JOIN coffee.gold.dim_menu_items mi
  ON ti.item_id = mi.item_id
WHERE mi.__END_AT IS NULL
GROUP BY ti.item_id, mi.item_name, mi.category
ORDER BY total_sales DESC
LIMIT 3;

In [0]:
--------------------------------------------------------
-- AGG 3: VOUCHER USAGE SUMMARY
--
-- Business KPI:
--   - Voucher adoption: voucher vs no voucher
--   - Sales and discount impact
--------------------------------------------------------
CREATE OR REFRESH MATERIALIZED VIEW  coffee.gold.agg_voucher_usage
COMMENT "Voucher usage KPIs derived from fact_transactions"
AS
SELECT
  CASE
    WHEN voucher_id IS NULL THEN 'NO_VOUCHER'
    ELSE 'VOUCHER_USED'
  END AS voucher_flag,
  COUNT(DISTINCT transaction_id) AS total_transactions,
  ROUND(SUM(final_amount), 2) AS total_sales,
  ROUND(SUM(discount_applied), 2) AS total_discount
FROM coffee.gold.fact_transactions
GROUP BY
  CASE
    WHEN voucher_id IS NULL THEN 'NO_VOUCHER'
    ELSE 'VOUCHER_USED'
  END;

In [0]:
--------------------------------------------------------
-- AGG 4: PAYMENT METHOD SPLIT
--
-- Business KPI:
--   - Payment preference distribution
--   - Total sales and transactions per method
--------------------------------------------------------
CREATE OR REFRESH  MATERIALIZED VIEW  coffee.gold.agg_payment_method_split
COMMENT "Sales and transaction split by payment method"
AS
SELECT
  t.payment_method_id,
  pm.method_name,
  pm.category,
  COUNT(DISTINCT t.transaction_id) AS total_transactions,
  ROUND(SUM(t.final_amount), 2) AS total_sales,
  ROUND(AVG(t.final_amount), 2) AS avg_order_value
FROM coffee.gold.fact_transactions t
JOIN coffee.gold.dim_payment_methods pm
  ON t.payment_method_id = pm.method_id
WHERE pm.__END_AT IS NULL
GROUP BY
  t.payment_method_id,
  pm.method_name,
  pm.category;

In [0]:
--------------------------------------------------------
-- AGG 5: CUSTOMER LIFETIME VALUE (CLV)
--
-- Business KPI:
--   - Total spend per customer
--   - Total orders per customer
--   - Average order value per customer
--   - First and last order date
--------------------------------------------------------
CREATE OR REFRESH  MATERIALIZED VIEW  coffee.gold.agg_customer_clv
COMMENT "Customer lifetime value metrics derived from fact_transactions"
AS
SELECT
  t.user_id,
  COUNT(DISTINCT t.transaction_id) AS total_orders,
  ROUND(SUM(t.final_amount), 2) AS total_spend,
  ROUND(AVG(t.final_amount), 2) AS avg_order_value,
  MIN(t.created_at) AS first_order_date,
  MAX(t.created_at) AS last_order_date
FROM coffee.gold.fact_transactions t
GROUP BY t.user_id;

In [0]:
--------------------------------------------------------
-- AGG 6: STORE PERFORMANCE
--
-- Business KPI:
--   - Sales and transactions per store
--   - Average order value per store
--   - Region breakdown using store city/state
--------------------------------------------------------
CREATE OR REFRESH  MATERIALIZED VIEW  coffee.gold.agg_store_performance
COMMENT "Store performance KPIs derived from fact_transactions joined with current dim_stores"
AS
SELECT
  t.store_id,
  s.store_name,
  s.city,
  s.state,
  COUNT(DISTINCT t.transaction_id) AS total_transactions,
  ROUND(SUM(t.final_amount), 2) AS total_sales,
  ROUND(AVG(t.final_amount), 2) AS avg_order_value,
  ROUND(SUM(t.discount_applied), 2) AS total_discount
FROM coffee.gold.fact_transactions t
JOIN coffee.gold.dim_stores s
  ON t.store_id = s.store_id
WHERE s.__END_AT IS NULL
GROUP BY
  t.store_id,
  s.store_name,
  s.city,
  s.state;

In [0]:
--------------------------------------------------------
-- AGG 7: TOP 10 CUSTOMERS BY SPEND
--
-- Business KPI:
--   - Identify high value customers
--------------------------------------------------------
CREATE OR REFRESH  MATERIALIZED VIEW  coffee.gold.agg_top_10_customers_by_spend
COMMENT "Top 10 customers ranked by total spend"
AS
SELECT
  t.user_id,
  COUNT(DISTINCT t.transaction_id) AS total_orders,
  ROUND(SUM(t.final_amount), 2) AS total_spend,
  ROUND(AVG(t.final_amount), 2) AS avg_order_value
FROM coffee.gold.fact_transactions t
GROUP BY t.user_id
ORDER BY total_spend DESC
LIMIT 10;

In [0]:
-- =========================================================
-- AGG: CUSTOMER CHURN LIST (6 MONTHS)
--
-- Definition:
--   A customer is churned if they have NOT made a transaction
--   in the last 6 months.
-- =========================================================

CREATE OR REFRESH MATERIALIZED VIEW coffee.gold.agg_customer_churn_6m
COMMENT "Churn list: customers with no transactions in last 6 months"
AS
WITH customer_last_txn AS (
  SELECT
    user_id,
    MAX(created_at) AS last_txn_ts
  FROM coffee.gold.fact_transactions
  WHERE user_id IS NOT NULL
  GROUP BY user_id
)
SELECT
  user_id,
  last_txn_ts,
  CASE
    WHEN last_txn_ts < add_months(current_timestamp(), -6) THEN 1
    ELSE 0
  END AS is_churned_6m
FROM customer_last_txn;


In [0]:
-- =========================================================
-- AGG: NEW CUSTOMER CHURN (ONE-TIME BUYERS)
--
-- Definition:
--   A customer is churned if:
--     1) They have made only 1 transaction total
--     2) Their only transaction happened more than 6 months ago
--
-- Business KPI:
--   Identifies one-time customers who never returned.
-- =========================================================

CREATE OR REFRESH MATERIALIZED VIEW coffee.gold.agg_new_customer_churn_6m
COMMENT "New customer churn: one-time buyers whose only purchase was > 6 months ago"
AS
WITH customer_summary AS (
  SELECT
    user_id,
    COUNT(DISTINCT transaction_id) AS total_orders,
    MAX(created_at) AS last_txn_ts
  FROM coffee.gold.fact_transactions
  WHERE user_id IS NOT NULL
  GROUP BY user_id
)
SELECT
  user_id,
  total_orders,
  last_txn_ts,
  CASE
    WHEN total_orders = 1
     AND last_txn_ts < add_months(current_timestamp(), -6)
    THEN 1
    ELSE 0
  END AS is_new_customer_churned_6m
FROM customer_summary;
